# Collecte automatisée en python d'un site en 2025-2026

- dates de révision :

## Binôme / trinôme: 
- nom1 prenom1 adresse UBS couriel1 (implication dans collecte : - [ ] Oui - [ ] Non) ;
- nom2 prenom2 adresse UBS couriel2 (implication dans collecte : - [ ] Oui - [ ] Non).

## Expérience préalable dans la collecte automatisée
Si impliqué dans la collecte, (reproduire le paragraphe si aussi deuxième binôme)
- nom1 prenom1 : [ ] j'ai refait voire reproduit et étudier attentivement les exemples du cours concernant :
    - [ ] `urllib` - [ ] `requests` - [ ] `re` - [ ] `lxml` - [ ] `bs4` - [ ] scrapy - [ ] `selenium` - [ ] scrapy-selenium 
- J'ai étudié exhaustivement la documentation officielle de :
    - [ ] `urllib` - [ ] `requests` - [ ] `re` - [ ] `lxml` - [ ] `bs4` - [ ] scrapy - [ ] `selenium` - [ ] scrapy-selenium 
- J'ai déjà collecté des informations sur X sites utilisant simplement html comme moteur de rendu de l'information (voir les fiches Y, Z)
- J'ai déjà collecté des informations sur X sites utilisant javascript / AJAX comme moteur de rendu de l'information (voir les fiches Y, Z)
- J'ai déjà été confronté à des mesures de rétorsion / défense de sites comme :
    - [ ] banissement IP - [ ] Captcha  - [ ] Obligation de s'identifier via un compte - [ ] Autre : préciser

## Site :
- URL :
- Description du site :
- Date de l'étude de ce site : 
- Technologie de rendu des informations clés à récupérer suite à l'analyse préalable : - [ ] HTML - [x] Javascript

## Analyse préalable :
- Qualification du cas : - [ ] Simple - [ ] Complexe 
- Outils utilisés : - [ ] Inspecter de FireFox - [x] Inspecter de Chrome - [ ] scrapy shell
- Sélecteurs utilisés donc envisagés pour la programmation : - [ ] CSS - [ ] Xpath
- Modules python pour la collecte de document : - [ ] `urllib` - [ ] `requests` - [ ] `selenium` - [ ] j'utilise srapy 
- Modules python pour l'analyse de document : - [ ] `re` - [ ] `lxml` - [ ] `bs4` - [ ] `selenium` - [ ] j'utilise srapy

- Pour chacune des informations recherchées, décrire en français votre stratégie de collecte puis exprimez cette stratégie sous forme opérationnelle (règle, expression bs4, expression réguliere, utilisation de fonction dans la scrapy shell, etc.
    - `ID_Produit`, éventuellement ;
    - `Libelle_produit` ;
    - `Description_produit` ;
    - `Prix_unitaire_HT_produit` ;
    - `code_barre_produit`, éventuellement ;
    - `categorie_produit` ;
    - `sous_categorie_produit` ;
    - `famille_produit`.

## Programmation de la collecte :
Pour toutes les technologies utilisées, on a vu qu'elles pouvaient être lancée depuis un notebook jupyter. Par contre pour une programmation pro scrapy exige un projet bien structuré. Il s'agit là de donner un programme qui ne récupère que les dix premiers exemplaires des informations recherchées. Mettre un lien vers l'archive contenant les programmes qui font la collecte exhaustive.

La collecte s'est elle déroulée en plusieurs fois ? - [ ] non - [ ] oui

Si non, date de la collecte exhaustive et nombre de produits collectés :

Si oui, dates et pour chacune d'elles nombre de produits collectés :


### Programme qui récupère les dix premers produits

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
options.binary_location = "D:\\tools\\GoogleChromePortable\\GoogleChromePortable.exe"
options.add_argument('--no-sandbox')
options.add_argument('--no-default-browser-check')
options.add_argument('--no-first-run')
options.add_argument('--disable-gpu')
options.add_argument('--disable-extensions')
options.add_argument('--disable-default-apps')
options.add_argument("--remote-debugging-port=9222")

driver = webdriver.Chrome("D:\\tools\\WinPython-64bit-3.5.3.1Qt5\\python-3.5.3.amd64\\selenium_env\\chromedriver.exe", options=options)

driver.get('https://www.electrodepot.fr/tv-image-son/television.html')

data = []  # Liste pour stocker les dicts de chaque produit
#sous_categorie = {"TV < 108 cm":1,"TV de 108 à 146 cm":2,"TV de 147 à 178 cm":3,"TV de plus de 179 cm":4,
                 #"Vidéoprojecteur, écran":5}
    
i = 1
try:
    WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, 'productlist-item_block'))
    )
    
    product_links = []
    elements = driver.find_elements(By.XPATH, '//div[contains(@class, "productlist-item_block")]/a[@href]')
    
    for element in elements:
        href = element.get_attribute('href')
        if href and href.endswith('.html'):
            product_links.append(href)
        if len(product_links) == 10:
            break
    
    for link in product_links:
        CATEGORIE_PRODUIT = "Vidéo"
        driver.get(link)
        
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.page-title'))
        )
        
        # Extraction des infos
        try:
            name = driver.find_element(By.CSS_SELECTOR, 'h1.page-title').text.strip()
        except:
            name = None
        
        try:
            price_element = driver.find_element(By.CSS_SELECTOR, '.price-wrapper .price')
            price_whole = price_element.text.strip().split('€')[0]
            price_fraction = ""
            if 'sup' in price_element.get_attribute('outerHTML'):
                price_fraction = price_element.find_element(By.CSS_SELECTOR, 'sup span').text.strip()
            price = price_whole + price_fraction if price_fraction else price_whole
        except:
            price = None
        
        try:
            rating = driver.find_element(By.CSS_SELECTOR, 'div[itemprop="ratingValue"]').text.strip()
        except:
            rating = None

        # Récupération specs dans un dict
        specs = {}
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, 'product-attribute-specs-table'))
            )
            table = driver.find_element(By.ID, 'product-attribute-specs-table')
            rows = table.find_elements(By.TAG_NAME, 'tr')
            for row in rows:
                cells = row.find_elements(By.TAG_NAME, 'td')
                if len(cells) == 2:
                    key = cells[0].text.strip()
                    value = cells[1].text.strip()
                    specs[key] = value
        except:
            pass
                # Récupération de la catégorie (ex: "TV de 108 à 146 cm")
        try:
            category_element = driver.find_element(By.CSS_SELECTOR, 'a.prodItem__topLink-back-link span')
            category = category_element.text.strip()
        except:
            category = None

        
        # Construire dict ligne avec les infos souhaitées
        product_data = {"ID_PRODUIT":i,
                        "CODE_PRODUIT": specs.get("Code article"),
                        #"Taille écran en cm" : specs.get("Taille écran en cm"),
            "LIBELLE": name,
            "PRIX": price,
            #"NOTE": rating,
            "ID_ENTREPOT": specs.get("Adresse postale"),
            "CATEGORIE_PRODUIT":CATEGORIE_PRODUIT,
            "SOUS_CATEGORIE": category

        } 
        
        
        data.append(product_data)
        i = i+1

except Exception as e:
    print("Erreur lors du scraping :", e)

driver.quit()

# Conversion en DataFrame pandas
df = pd.DataFrame(data)
ordered_columns = ['ID_PRODUIT', 'CODE_PRODUIT', 'LIBELLE', 'PRIX', 'ID_ENTREPOT', 'CATEGORIE_PRODUIT', 'SOUS_CATEGORIE']
df = df[ordered_columns]

# Affichage ou sauvegarde
print(df)

# Par exemple sauvegarder dans un CSV
df.to_csv('produits_electrodepot.csv', index=False)

Cette fiche doit être complétée et présente dans l'archive de rendu au niveau du répertoire `4-Scrap`